# Медианный фильтр для шума «соль и перец» на GPU

Исправленная версия ноутбука: формат ввода пути к изображению оставлен через `input(...)`, но чтение и сохранение изображений выполняются с поддержкой русских букв в пути.

In [ ]:
from numba import cuda
import numpy as np
import cv2
import time
import math
from pathlib import Path

In [ ]:
def read_image_unicode(filename, flags=cv2.IMREAD_GRAYSCALE):
    
    filename = str(filename)
    if not Path(filename).exists():
        return None
    data = np.fromfile(filename, dtype=np.uint8)
    image = cv2.imdecode(data, flags)
    return image


def write_image_unicode(filename, image):
    
    filename = str(filename)
    ext = Path(filename).suffix
    if ext == "":
        raise ValueError("У файла должно быть расширение, например .bmp или .png")
    ok, encoded = cv2.imencode(ext, image)
    if not ok:
        return False
    encoded.tofile(filename)
    return Path(filename).exists()

In [ ]:
def add_salt_and_pepper_noise(image, salt_prob, pepper_prob):

    noisy_image = np.copy(image)
    total_pixels = image.size

    # Добавляем соль
    num_salt = np.ceil(salt_prob * total_pixels).astype(int)
    coords = [np.random.randint(0, i - 1, num_salt) for i in image.shape]
    noisy_image[coords[0], coords[1]] = 255  # Соль - белые пиксели

    # Добавляем перец
    num_pepper = np.ceil(pepper_prob * total_pixels).astype(int)
    coords = [np.random.randint(0, i - 1, num_pepper) for i in image.shape]
    noisy_image[coords[0], coords[1]] = 0  # Перец - черные пиксели

    return noisy_image

In [ ]:
@cuda.jit
def mf_kernel(input_image, output_image, width, height):
  
    # Определяем позицию потока в сетке
    x, y = cuda.grid(2)

    if x < 1 or y < 1 or x >= width - 1 or y >= height - 1:
        # Пропускаем граничные пиксели (нельзя применить 3x3 фильтр)
        return

    # Создаем список для хранения 9 пикселей из области 3x3
    window = cuda.local.array(9, dtype=np.float32)

    idx = 0
    for j in range(-1, 2):
        for i in range(-1, 2):
            window[idx] = input_image[y + j, x + i]
            idx += 1

    # Сортировка пузырьком для нахождения медианы
    for i in range(9):
        for j in range(i + 1, 9):
            if window[i] > window[j]:
                temp = window[i]
                window[i] = window[j]
                window[j] = temp

    # Присваиваем медианное значение выходному изображению
    output_image[y, x] = window[4]

In [ ]:
def mf_gpu(input_image):
  
    # Преобразуем входное изображение в float32 для совместимости
    input_image = input_image.astype(np.float32)

    # Получаем размеры изображения
    height, width = input_image.shape

    # Выделяем память для выходного изображения
    output_image = np.zeros((height, width))

    # Копируем данные на устройство
    cuda_input = cuda.to_device(input_image)
    cuda_output = cuda.to_device(output_image)

    # Определяем размеры блока и сетки
    threads_per_block = (16, 16)
    blocks_per_grid_x = math.ceil(width / threads_per_block[0])
    blocks_per_grid_y = math.ceil(height / threads_per_block[1])
    blocks_per_grid = (blocks_per_grid_x, blocks_per_grid_y)

    # Запускаем ядро
    mf_kernel[blocks_per_grid, threads_per_block](cuda_input, cuda_output, width, height)
    cuda.synchronize()

    # Копируем результат обратно на хост
    output_image = cuda_output.copy_to_host()

    return output_image

In [ ]:
def save_image(image, filename):

    # Преобразуем изображение в формат uint8
    image = np.clip(image, 0, 255).astype(np.uint8)
    if not write_image_unicode(filename, image):
        print(f"Ошибка: Не удалось сохранить изображение в файл '{filename}'.")

## Запуск программы

Формат ввода изображения сохранён таким же, как в исходном файле.

In [ ]:
# Запрос пути к файлу у пользователя
file_path = input("путь к изображению: ")

# Запрос у пользователя, хочет ли он зашумить изображение
choice = input("Введите 1 для зашумления изображения или 0 для выбора зашумленного файла: ")

if choice == '1':
    # Загружаем изображение
    src_image = read_image_unicode(file_path, cv2.IMREAD_GRAYSCALE)

    # Проверка на успешную загрузку изображения
    if src_image is None:
        print(f"Ошибка: Не удалось загрузить изображение из файла '{file_path}'. Проверьте путь и формат файла.")
    else:
        # Добавляем шум
        noisy_image = add_salt_and_pepper_noise(src_image, salt_prob=0.05, pepper_prob=0.05)
        
        # Сохраняем зашумленное изображение
        save_image(noisy_image, 'noisy_image.bmp')
        print("Зашумленное изображение сохранено как 'noisy_image.bmp'.")

elif choice == '0':
    # Загружаем зашумленное изображение
    noisy_image = read_image_unicode(file_path, cv2.IMREAD_GRAYSCALE)

    if noisy_image is None:
        print(f"Ошибка: Не удалось загрузить изображение из файла '{file_path}'. Проверьте путь и формат файла.")
    else:
        # Применяем медианный фильтр
        start_gpu = time.time()
        filtered_image = mf_gpu(noisy_image)
        time_gpu = time.time() - start_gpu

        # Сохраняем результирующее изображение
        save_image(filtered_image, 'filtered_image.bmp')

        # Время
        print("Время (GPU): ", time_gpu)
        print("Отфильтрованное изображение сохранено как 'filtered_image.bmp'.")

else:
    print("Ошибка: нужно ввести 1 или 0.")